In [4]:
import pandas as pd
import re
import os
import sys
from sqlalchemy import text

projectRoot = os.path.abspath(os.path.join(os.getcwd(), '..'))
if projectRoot not in sys.path:
    sys.path.insert(0, projectRoot)
from helpers import initDB

In [87]:
def draftFixtures():
    xl = pd.read_excel('draftFixtures.xlsx', sheet_name='Fixtures')
    xl.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
    xl = xl.drop(columns=['Unnamed: 1'])
    games = []
    headers = xl.columns[1:]
    for header in headers:
        #account for joint columns
        if '/' in header:
            teams = header.split('/')
        else:
            teams = [header]
        for team in teams:
            teamSeries = xl[header]
            for idx, game in enumerate(teamSeries):
                if pd.isna(game):
                    continue
                gameData = [xl['Date'].iloc[idx], team, idx+1]
                if game == 'BYE' or game == 'Bye':
                    gameData.append('BYE')
                elif game == 'Finals':
                    gameData.append('Finals')
                else:
                    #opponent
                    gameData.append(re.search(r"\(([^)]*)\)", game).group(1)) 
                    #venue
                    if game.split(' (')[0] == 'Home':
                        gameData.append(game[-4:])
                    else:
                        gameData.append(game.split(' (')[0])
                    #times
                    times = game.split(' - ')[1].split(', ')
                    if len(teams) == 1:
                        if ' ' in times[0]:
                            gameData.append(times[0].split(' ')[0])
                        else:
                            gameData.append(times[0])
                    elif len(teams) == 2:
                        if team == teams[0]:
                            gameData.append(times[1].split(' ')[0])
                        elif team == teams[1]:
                            gameData.append(times[0])
                games.append(gameData)
    df = pd.DataFrame(games, columns=['Date', 'Team', 'Round', 'Opponent', 'Venue', 'Time'])
    #typo fixes
    def typoFix(team, round, target, value):
        mask = (df['Team']==team) & (df['Round']==round)
        df.loc[mask, target] = value
    typoFix('A Women', 20, 'Venue', 'MCG1')
    typoFix('A-Res Women', 17, 'Venue', 'MCG1')
    typoFix('D', 1, 'Venue', 'MCG2')
    typoFix('D-Res', 1, 'Venue', 'MCG2')
    df.loc[df['Venue'].isin(['30am', '25pm', '20pm', '05pm']), 'Venue'] = 'MCG1'
    engine = initDB()
    df.to_sql('rawfixtures', con=engine, if_exists='replace', index=False)
    return df

draftFixtures()

,Date,Team,Round,Opponent,Venue,Time
0,2026-04-11,A,1,Curtin Wesley,South Oval,3:05pm
1,2026-04-18,A,2,North Beach,MCG1,12.25pm
2,2026-04-25,A,3,Baldivis,Baldivis District Sporting Complex,3:05pm
3,2026-05-02,A,4,CBC,MCG1,2:20pm
4,2026-05-09,A,5,TA's,Bill Grayden,3:10pm
...,...,...,...,...,...,...
201,2026-08-01,Integrated,17,Wembley,Pat Goodridge,1pm
202,2026-08-08,Integrated,18,Finals,NaN,NaN
203,2026-08-15,Integrated,19,Finals,NaN,NaN
204,2026-08-22,Integrated,20,Finals,NaN,NaN


In [124]:
import numpy as np

def draftFixturesClean():
    engine = initDB()
    with engine.connect() as conn:
        rawDF = pd.read_sql(text("SELECT * FROM rawfixtures"), conn)
    df = rawDF.copy()
    #home team
    df['HomeTeam'] = np.select(
        [(df['Venue']=='MCG1')|(df['Venue']=='MCG2'), (df['Opponent']=='BYE')|(df['Opponent']=='Finals')],
        ['Universty', np.nan],
        default=df['Opponent']
    )
    #away team
    df['AwayTeam'] = np.select(
        [(df['Venue']=='MCG1')|(df['Venue']=='MCG2'), (df['Opponent']=='BYE')|(df['Opponent']=='Finals')],
        [df['Opponent'], np.nan],
        default='University'
    )
    #time
    df['Time'] = df['Time'].str.replace('.', ':', regex=False)
    t1 = pd.to_datetime(df['Time'], format='%I:%M%p', errors='coerce')
    t2 = pd.to_datetime(df['Time'], format='%I:%p', errors='coerce')
    df['Time'] = t1.fillna(t2).dt.time
    #date
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.date
    #uni team
    teamMap = {'A': 'A', 'A-Res': 'AR', 'A-Res Women': 'ARW', 'C-Colts':'BJC', 'D': 'PIR', 'D-Res': 'DIN', 'A-Colts': 'PSC', 'A Women': 'WA', 'Integrated': 'INT'}
    df['UniTeam'] = df['Team'].map(teamMap)
    #venue url
    df['VenueURL'] = ''
    #round
    df['Round'] = np.where(df['Opponent'] == 'Finals', 'Finals', df['Round'])
    df['Round'] = df['Round'].apply(lambda x: f'Round {x}' if x != 'Finals' else 'Finals')
    finalsMask = df['Opponent']=='Finals'
    df.loc[finalsMask, 'FinalsNo'] = (df[finalsMask].sort_values(['UniTeam', 'Date']).groupby('UniTeam').cumcount()+1)
    df.loc[finalsMask, 'Round'] = ('Final ' + df.loc[finalsMask, 'FinalsNo'].astype(int).astype(str))
    df.drop(columns=['FinalsNo'], inplace=True)
    #gameID
    df['GameID'] = df['UniTeam'] + ':' + df['Round'] + ':' + '26'
    #umpires
    df['Central'] = np.nan
    df['Boundary'] = np.nan
    df['Goal'] = np.nan
    #season df
    seasonDF = df[['Round', 'HomeTeam', 'AwayTeam', 'Venue', 'VenueURL', 'Date', 'Time', 'GameID', 'UniTeam', 'Central', 'Boundary', 'Goal']].copy()
    #preseason df
    preseasonGames = [
        ['Preseason 1', 'University', 'Trinity Aquinas', 'MCG1', '', '2026-03-21', '09:00', 'BJC: Preseason 1:26', 'BJC', np.nan, np.nan, np.nan],
        ['Preseason 1', 'University', 'Trinity Aquinas', 'MCG1', '', '2026-03-21', '11:00', 'PSC: Preseason 1:26', 'PSC', np.nan, np.nan, np.nan],
        ['Preseason 1', 'University', 'Trinity Aquinas', 'MCG1', '', '2026-03-21', '13:00', 'AR: Preseason 1:26', 'AR', np.nan, np.nan, np.nan],
        ['Preseason 1', 'University', 'Trinity Aquinas', 'MCG1', '', '2026-03-21', '15:00', 'A: Preseason 1:26', 'A', np.nan, np.nan, np.nan],

        ['Preseason 1', 'University', 'Trinity Aquinas', 'MCG2', '', '2026-03-21', '09:00', 'ARW: Preseason 1:26', 'ARW', np.nan, np.nan, np.nan],
        ['Preseason 1', 'University', 'Trinity Aquinas', 'MCG2', '', '2026-03-21', '10:40', 'WA: Preseason 1:26', 'WA', np.nan, np.nan, np.nan],
        ['Preseason 1', 'University', 'Trinity Aquinas', 'MCG2', '', '2026-03-21', '12:25', 'DIN: Preseason 1:26', 'Din', np.nan, np.nan, np.nan],
        ['Preseason 1', 'University', 'Trinity Aquinas', 'MCG2', '', '2026-03-21', '13:40', 'PIR: Preseason 1:26', 'PIR', np.nan, np.nan, np.nan],
    ]
    preSeasonDF = pd.DataFrame(preseasonGames, columns=['Round', 'HomeTeam', 'AwayTeam', 'Venue', 'VenueURL', 'Date', 'Time', 'GameID', 'UniTeam', 'Central', 'Boundary', 'Goal'])
    preSeasonDF['Date'] = pd.to_datetime(preSeasonDF['Date'], errors='coerce').dt.date
    preSeasonDF['Time'] = pd.to_datetime(preSeasonDF['Time'], format='%H:%M', errors='coerce').dt.time
    #final df
    finalDF = pd.concat([seasonDF, preSeasonDF], ignore_index=True)
    finalDF = finalDF.sort_values(by=['UniTeam', 'Date'], ignore_index=True)
    with engine.begin() as conn:
        conn.execute(text('TRUNCATE TABLE fixtures'))
    finalDF.to_sql('fixtures', con=engine, if_exists='append', index=False)
    return finalDF

draftFixturesClean()

,Round,HomeTeam,AwayTeam,Venue,VenueURL,Date,Time,GameID,UniTeam,Central,Boundary,Goal
0,Preseason 1,University,Trinity Aquinas,MCG1,,2026-03-21,15:00:00,A: Preseason 1:26,A,NaN,NaN,NaN
1,Round 1,Curtin Wesley,University,South Oval,,2026-04-11,15:05:00,A:Round 1:26,A,NaN,NaN,NaN
2,Round 2,Universty,North Beach,MCG1,,2026-04-18,12:25:00,A:Round 2:26,A,NaN,NaN,NaN
3,Round 3,Baldivis,University,Baldivis District Sporting Complex,,2026-04-25,15:05:00,A:Round 3:26,A,NaN,NaN,NaN
4,Round 4,Universty,CBC,MCG1,,2026-05-02,14:20:00,A:Round 4:26,A,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
209,Round 20,Universty,TA's,MCG1,,2026-08-22,09:30:00,WA:Round 20:26,WA,NaN,NaN,NaN
210,Final 1,NaN,NaN,NaN,,2026-08-29,NaT,WA:Final 1:26,WA,NaN,NaN,NaN
211,Final 2,NaN,NaN,NaN,,2026-09-05,NaT,WA:Final 2:26,WA,NaN,NaN,NaN
212,Final 3,NaN,NaN,NaN,,2026-09-12,NaT,WA:Final 3:26,WA,NaN,NaN,NaN
